# 02 · FONDTRIM + FONDCART: cleaning and selection calibration

**Objective:** measure what the two cumulative sources (`fund_quarterly`, `fund_holdings`) add to the monthly tables, and calibrate the cleaning and selection thresholds.

**Scope.** The four Silver tables: `fund_nav_daily` + `fund_metadata` (mass sources) and the two cumulative snapshots, whose six periods cover months 06 and 12 only. `as_of_period` is pinned to 202605 within the notebook.


In [1]:
import duckdb

ROOT = "../.."  # repo root, relative to this notebook's folder (notebooks/exploration)
duckdb.sql(f"""CREATE OR REPLACE TEMP VIEW nav AS
    SELECT * FROM read_parquet('{ROOT}/data/silver/fund_nav_daily/**/*.parquet', hive_partitioning=true)""")
duckdb.sql(f"""CREATE OR REPLACE TEMP VIEW meta AS
    SELECT * FROM read_parquet('{ROOT}/data/silver/fund_metadata/**/*.parquet', hive_partitioning=true)""")
duckdb.sql(f"""CREATE OR REPLACE TEMP VIEW q AS
    SELECT * FROM read_parquet('{ROOT}/data/silver/fund_quarterly/**/*.parquet', hive_partitioning=true)""")
duckdb.sql(f"""CREATE OR REPLACE TEMP VIEW h AS
    SELECT * FROM read_parquet('{ROOT}/data/silver/fund_holdings/**/*.parquet', hive_partitioning=true)""")

# 36-month trailing window; as_of_period is the last published month (202605)
duckdb.sql("""
    CREATE OR REPLACE TEMP VIEW wnd AS
    SELECT max(value_date) AS as_of_date,
           max(value_date) - INTERVAL 36 MONTH + INTERVAL 1 DAY AS window_start
    FROM nav WHERE FechaDatos = '202605'
""")
duckdb.sql("""
    CREATE OR REPLACE TEMP VIEW active AS
    SELECT n.* FROM nav n
    WHERE n.FechaDatos = '202605'
      AND n.value_date = (SELECT as_of_date FROM wnd)
      AND Patrimonio > 0 AND Participes > 0
""")
print("Reproducibility header: slice of Silver measured in this run")
display(
    duckdb.sql("""
    SELECT (SELECT count(*) FROM nav)   AS nav_rows,
           (SELECT count(*) FROM meta)  AS meta_rows,
           (SELECT count(*) FROM q)     AS quarterly_rows,
           (SELECT count(*) FROM h)     AS holdings_rows,
           (SELECT min(FechaDatos) || '..' || max(FechaDatos) FROM nav) AS nav_periods,
           (SELECT as_of_date::VARCHAR FROM wnd) || ' / '
              || (SELECT window_start::VARCHAR FROM wnd) AS as_of_start,
           current_date                 AS executed_on
""").df()
)

Reproducibility header: slice of Silver measured in this run


,nav_rows,meta_rows,quarterly_rows,holdings_rows,nav_periods,as_of_start,executed_on
0,3806718,125807,18215,572263,202301..202605,2026-05-31 / 2023-06-01 00:00:00,2026-10-25


## 1. Universe and publication cadence
**Question:** how often does the source publish, and how large is the investable universe at the as-of date? A class is active when the last published day of 202605 (2026-05-31) carries positive AUM and a positive holder count; every later section refers to this universe.

In [2]:
print("publication cadence per Silver table")
display(
    duckdb.sql("""
    SELECT 'fund_nav_daily' AS table_name, count(DISTINCT FechaDatos) AS periods,
           min(FechaDatos) AS first_period, max(FechaDatos) AS last_period,
           count(*) AS rows
    FROM nav
    UNION ALL SELECT 'fund_metadata', count(DISTINCT FechaDatos),
           min(FechaDatos), max(FechaDatos), count(*)  FROM meta
    UNION ALL SELECT 'fund_quarterly', count(DISTINCT FechaDatos),
           min(FechaDatos), max(FechaDatos), count(*) FROM q
    UNION ALL SELECT 'fund_holdings', count(DISTINCT FechaDatos),
           min(FechaDatos), max(FechaDatos), count(*) FROM h
    ORDER BY 1
""").df()
)
print("growth across the six snapshot closures (aligned months only)")
display(
    duckdb.sql("""
    SELECT qc.FechaDatos AS periodo, qc.classes, hc.positions
    FROM (SELECT FechaDatos, count(*) AS classes FROM q GROUP BY 1) qc
    JOIN (SELECT FechaDatos, count(*) AS positions FROM h GROUP BY 1) hc
      ON hc.FechaDatos = qc.FechaDatos
    ORDER BY 1
""").df()
)

publication cadence per Silver table


,table_name,periods,first_period,last_period,rows
0,fund_holdings,6,202306,202512,572263
1,fund_metadata,41,202301,202605,125807
2,fund_nav_daily,41,202301,202605,3806718
3,fund_quarterly,6,202306,202512,18215


growth across the six snapshot closures (aligned months only)


,periodo,classes,positions
0,202306,2982,89659
1,202312,3004,90087
2,202406,3014,93600
3,202412,3055,96537
4,202506,3091,99250
5,202512,3069,103130


In [3]:
print("active classes, compartments, entities and AUM at the as-of date")
display(
    duckdb.sql("""
    SELECT count(*) AS active_classes,
           count(DISTINCT (Tipo, NumeroRegistro, NumeroCompartimento)) AS active_compartimentos,
           count(DISTINCT (Tipo, NumeroRegistro)) AS active_entities,
           round(sum(Patrimonio) / 1e9, 1) AS total_aum_bn
    FROM active
""").df()
)
print("compartments per entity (active universe)")
display(
    duckdb.sql("""
    WITH comp AS (
        SELECT Tipo, NumeroRegistro,
               count(DISTINCT NumeroCompartimento) AS n
        FROM active GROUP BY 1, 2)
    SELECT count(*) AS entities,
           sum(CASE WHEN n = 1 THEN 1 ELSE 0 END) AS single_compartment,
           sum(CASE WHEN n BETWEEN 2 AND 5 THEN 1 ELSE 0 END) AS two_to_five,
           sum(CASE WHEN n > 5 THEN 1 ELSE 0 END) AS more_than_five
    FROM comp
""").df()
)

active classes, compartments, entities and AUM at the as-of date


,active_classes,active_compartimentos,active_entities,total_aum_bn
0,2912,1631,1430,485.5


compartments per entity (active universe)


,entities,single_compartment,two_to_five,more_than_five
0,1430,1381.0,31.0,18.0


- **Note:** cadence and universe in three families.
  - Monthly mass tables: `fund_nav_daily` and `fund_metadata` publish one period per month, 41 periods from 202301 to 202605.
  - The snapshots (`fund_quarterly`, `fund_holdings`) ship only at months 06 and 12: six closures, with rows growing from 2.982 to 3.069 classes and 89.659 to 103.130 positions. An earlier working assumption of quarterly closures was wrong.
  - Universe at the as-of date: 2.912 active classes across 1.631 compartments of 1.430 entities, 485,5 bn € of assets. Only 49 entities (3,4 %) run more than one compartment (31 with 2 to 5, 18 with more than 5); the rest are single-compartment funds with compartment index 0.
- **Hypothesis:** the semiannual cadence is a publisher property, not a data defect; windows built on the snapshots must expect two data points per year.
- **Gate:** the snapshot parsers enforce months 06 and 12 (`trim_period_months` in the gate). The 36-month window holds 1.095 days over 35,3 published months (157 theoretical weekly buckets), which caps weekly coverage at about 0,98; the calibrated `min_coverage` of [§4](#4-weekly-series-calibration) must tolerate that ceiling.

## 2. Snapshot coverage, currencies and vocations
**Question:** do the snapshot tables join cleanly onto the monthly universe, and what do they add (currency labels, vocations)? The selection reads both attributes from the snapshots, so join quality and label levels decide how much of the universe can be ranked.

In [4]:
print("join directions between the snapshot and the monthly table (202512)")
display(
    duckdb.sql("""
    SELECT 'quarterly compartment without monthly row' AS direction, count(*) AS rows
    FROM (SELECT DISTINCT Tipo, NumeroRegistro, NumeroCompartimento
          FROM q WHERE FechaDatos = '202512'
          EXCEPT
          SELECT DISTINCT Tipo, NumeroRegistro, NumeroCompartimento
          FROM nav WHERE FechaDatos = '202512')
    UNION ALL
    SELECT 'monthly class without vocation', count(*)
    FROM (SELECT DISTINCT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase
          FROM nav WHERE FechaDatos = '202512'
          EXCEPT
          SELECT DISTINCT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase
          FROM q WHERE FechaDatos = '202512')
""").df()
)
print("currency codes at both levels (202512)")
display(
    duckdb.sql("""
    SELECT 'class level' AS level, COALESCE(CodigoDivisa, 'None') AS code,
           count(*) AS rows
    FROM q WHERE FechaDatos = '202512' GROUP BY 1, 2
    UNION ALL
    SELECT 'entity level', COALESCE(CodigoDivisaIIC, 'None'), count(*)
    FROM q WHERE FechaDatos = '202512' GROUP BY 1, 2
    ORDER BY 1, 3 DESC
""").df()
)
print(
    "cross-level disagreements:",
    duckdb.sql("""
        SELECT count(*) FROM q WHERE FechaDatos = '202512'
          AND CodigoDivisaIIC IS NOT NULL AND CodigoDivisa IS NOT NULL
          AND CodigoDivisa <> CodigoDivisaIIC
""").fetchall()[0][0],
)

join directions between the snapshot and the monthly table (202512)


,direction,rows
0,quarterly compartment without monthly row,0
1,monthly class without vocation,21


currency codes at both levels (202512)


,level,code,rows
0,class level,EUR,3067
1,class level,USD,1
2,class level,None,1
3,entity level,EUR,2657
4,entity level,None,411
5,entity level,USD,1


cross-level disagreements: 0


- **Note:** the join at 202512 is nearly perfect.
  - No quarterly compartment lacks a monthly row; only 21 monthly classes carry no vocación that day (they fall in the null group below).
  - Currency appears at two levels: class (`CodigoDivisa`, 3.067 of 3.069 rows as `EUR`, a single null) and entity (`CodigoDivisaIIC`, 411 nulls of 3.069), and the two levels never disagree when both are present.
- **Hypothesis:** the entity level is an optional attribute the publisher usually leaves blank while the class level is maintained, and the euro code is `EUR`.
- **Gate:** the selection reads currencies at the class level; the entity level stays available for per-compartment documentation.

In [5]:
duckdb.sql("""
    CREATE OR REPLACE TEMP VIEW comp512 AS
    SELECT Tipo, NumeroRegistro, NumeroCompartimento, VocacionInversora,
           sum(Patrimonio) AS aum,
           sum(NumeroParticipes) AS parts,
           bool_and(CodigoDivisa = 'EUR') AS all_eur
    FROM q WHERE FechaDatos = '202512'
    GROUP BY 1, 2, 3, 4
""")
print(
    "investable compartments per vocation (AUM at least 3 M€, at least 20 holders, all EUR)"
)
display(
    duckdb.sql("""
    SELECT VocacionInversora AS vocacion,
           count(*) AS compartimentos,
           sum(CASE WHEN aum >= 3e6 AND parts >= 20 AND all_eur
                    THEN 1 ELSE 0 END) AS investable
    FROM comp512 GROUP BY 1 ORDER BY investable DESC, 1
""").df()
)
print("ClaseFondo categories (top 6 of 12)")
display(
    duckdb.sql("""
    SELECT ClaseFondo, count(*) AS classes
    FROM q WHERE FechaDatos = '202512'
    GROUP BY 1 ORDER BY 2 DESC LIMIT 6
""").df()
)

investable compartments per vocation (AUM at least 3 M€, at least 20 holders, all EUR)


,vocacion,compartimentos,investable
0,Renta Variable Internacional,325,308.0
1,Renta Fija Euro,242,231.0
2,Global,198,192.0
3,None,171,163.0
4,Renta Variable Mixta Internacional,158,134.0
5,Renta Fija Mixta Internacional,131,109.0
6,Renta Fija Mixta Euro,103,94.0
7,Renta Fija Internacional,96,78.0
8,Renta Variable Euro,73,68.0
9,Garantizado de Rendimiento Fijo,47,47.0


ClaseFondo categories (top 6 of 12)


,ClaseFondo,classes
0,Otros,1666
1,Fondo que invierte mayoritariamente en otros f...,520
2,Fondo que invierte más del 10% en otros fondos...,470
3,Fondo que invierte más del 10% en otros fondos...,193
4,Fondo que invierte más del 10% en otros fondos...,44
5,Fondo que replica o reproduce un índice,43


- **Note:** 14 vocation groups appear in 202512, 13 named plus a null group.
  - All except `IIC de Gestión Pasiva` (2 compartments) clear the 5-peer minimum; the null group (171 compartments) reads as `no_vocation`.
  - The guaranteed families live in the vocation itself (`Garantizado de Rendimiento Fijo`, `Garantizado de Rendimiento Variable`).
  - `ClaseFondo` flags instead fund-of-funds and index-replica classes: about 1.270 classes declare exposure to other funds and 87 declare an index or index replica; the table shows the six largest of twelve categories.
- **Gate:** the peer-group mapping keeps all 13 named groups with at least 5 investable compartments; the guaranteed families leave the ranking by design (a maturity profile is not comparable on weekly dispersion), and the fund-of-funds or replica flags feed the documented look-through limitation instead of acting as a filter.

## 3. Official quarterly metrics
**Question:** what do the snapshot classes publish about expenses, fees and volatility, and is any of it directly usable as a filter or as a cross-check? One table per metric below, percentiles of the known values.

In [6]:
print("expenses, management fee and quarterly volatility by class (202512)")
display(
    duckdb.sql("""
    SELECT 'RatioTotalGastos' AS measure,
           count(*) - count(RatioTotalGastos_TrimestreActual) AS nulls,
           round(quantile_cont(RatioTotalGastos_TrimestreActual::DOUBLE, 0.05), 2) AS p05,
           round(quantile_cont(RatioTotalGastos_TrimestreActual::DOUBLE, 0.50), 2) AS p50,
           round(quantile_cont(RatioTotalGastos_TrimestreActual::DOUBLE, 0.95), 2) AS p95,
           sum(CASE WHEN RatioTotalGastos_TrimestreActual < 0 THEN 1 ELSE 0 END) AS negatives
    FROM q WHERE FechaDatos = '202512'
    UNION ALL
    SELECT 'ComisionGestion',
           count(*) - count(ComisionGestion),
           round(quantile_cont(ComisionGestion::DOUBLE, 0.05), 2),
           round(quantile_cont(ComisionGestion::DOUBLE, 0.50), 2),
           round(quantile_cont(ComisionGestion::DOUBLE, 0.95), 2),
           sum(CASE WHEN ComisionGestion < 0 THEN 1 ELSE 0 END)
    FROM q WHERE FechaDatos = '202512'
    UNION ALL
    SELECT 'Volatilidad_TrimestreActual',
           count(*) - count(Volatilidad_TrimestreActual),
           round(quantile_cont(Volatilidad_TrimestreActual::DOUBLE, 0.05), 2),
           round(quantile_cont(Volatilidad_TrimestreActual::DOUBLE, 0.50), 2),
           round(quantile_cont(Volatilidad_TrimestreActual::DOUBLE, 0.95), 2),
           sum(CASE WHEN Volatilidad_TrimestreActual < 0 THEN 1 ELSE 0 END)
    FROM q WHERE FechaDatos = '202512'
    ORDER BY 1
""").df()
)

expenses, management fee and quarterly volatility by class (202512)


,measure,nulls,p05,p50,p95,negatives
0,ComisionGestion,1,0.00,0.38,1.13,3.0
1,RatioTotalGastos,59,0.05,0.25,0.57,3.0
2,Volatilidad_TrimestreActual,132,0.07,3.95,14.08,0.0


- **Note:** the three metrics arrive almost complete.
  - `RatioTotalGastos`: 59 nulls of 3.069, median 0,25 %, p95 0,57 %, and 3 negative values (minimum -1,74 %) already reported by the gate's `trim_negative_metrics`.
  - `ComisionGestion`: median 0,38 %, a single null in 3.069 rows, and 3 negative values (minimum -1,02 %) in the same wind-down pattern.
  - `Volatilidad` of the quarter: median 3,95 with 132 nulls; the unit is not documented in the XSD.
- **Hypothesis:** negative expense ratios are wind-down rounding dust, the same pattern as the negative-AUM tail of `01_fondmens_fondregistro__nav_metadata_anomaly_profiling.ipynb`, and the volatility unit is percentage points, to confirm with the Documento Explicativo.
- **Gate:** a future fee filter can rank on `ComisionGestion` without extra parsing; the volatility cross-check against weekly dispersion waits for the unit being confirmed.

## 4. Weekly series calibration
**Question:** which quirks does the weekly return series carry for the *eligible* universe?

The series takes the last published day of each ISO week and computes returns on consecutive weeks only, so no return spans a gap and none overlap. `VL <= 0` rows stay out; their class-level anomaly is already measured elsewhere, and here they would only distort the dispersion metrics.

In [7]:
duckdb.sql("""
    CREATE OR REPLACE TEMP VIEW weekly_raw AS
    SELECT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase,
           date_trunc('week', value_date) AS week_start,
           max(value_date) AS week_end,
           arg_max(VL, value_date) AS VL_w
    FROM nav
    WHERE VL > 0
      AND EXTRACT(isodow FROM value_date) <= 5
      AND value_date <= (SELECT as_of_date FROM wnd)
      AND value_date >= (SELECT window_start FROM wnd)
    GROUP BY 1, 2, 3, 4, 5
""")
duckdb.sql("""
    CREATE OR REPLACE TEMP VIEW wkr AS
    SELECT *,
           CASE WHEN LAG(week_start) OVER cw = week_start - INTERVAL 7 DAY
                THEN VL_w::DOUBLE / LAG(VL_w::DOUBLE) OVER cw - 1 END AS ret_w
    FROM weekly_raw
    WINDOW cw AS (PARTITION BY Tipo, NumeroRegistro, NumeroCompartimento,
                  NumeroClase ORDER BY week_start)
""")
duckdb.sql("""
    CREATE OR REPLACE TEMP VIEW wkagg AS
    SELECT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase,
           count(*) AS n_weeks,
           count(*) - count(ret_w) AS n_gaps,
           sum(CASE WHEN ret_w = 0 THEN 1 ELSE 0 END) AS n_zero_ret,
           stddev_samp(ret_w) AS sd_w,
           max(abs(ret_w)) AS max_abs_ret,
           min(week_start) AS first_week
    FROM wkr GROUP BY 1, 2, 3, 4
""")
print(
    "per-class coverage (useful weeks / theoretical), active classes, buckets with >5 classes"
)
display(
    duckdb.sql("""
    SELECT round(n_weeks::DOUBLE / (SELECT ceil(date_diff('day', window_start, as_of_date) / 7.0) FROM wnd), 3)
           AS coverage,
           count(*) AS classes
    FROM wkagg JOIN active USING (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)
    GROUP BY 1 HAVING count(*) > 5
    ORDER BY 1
""").df()
)

per-class coverage (useful weeks / theoretical), active classes, buckets with >5 classes


,coverage,classes
0,0.032,9
1,0.076,14
2,0.083,6
3,0.096,7
4,0.102,6
5,0.121,7
6,0.153,6
7,0.185,6
8,0.204,6
9,0.210,6


- **Note:** weekly coverage is bimodal.
  - 2.211 classes publish through the whole 36-month window (coverage 1,00).
  - A long tail below 0,9 dominates the rest: classes launched inside the window and classes with interior holes; only 27 classes land in the thin edge between 0,9 and 1,0.
- **Hypothesis:** the eligible set is smaller than the 2.912 active classes, and the first observed week already separates short histories from coverage gaps inside that tail.
- **Gate:** `min_coverage` stays at 0,90; the partial window raises the theoretical ceiling to about 0,98, so no class is punished for calendar arithmetic.

In [8]:
print("zero-return week share per active class with at least 140 weeks; percentiles")
display(
    duckdb.sql("""
    SELECT round(quantile_cont(zero_share, 0.50), 4) AS p50,
           round(quantile_cont(zero_share, 0.90), 4) AS p90,
           round(quantile_cont(zero_share, 0.95), 4) AS p95,
           round(quantile_cont(zero_share, 0.99), 4) AS p99,
           round(quantile_cont(zero_share, 1.0), 4) AS p100
    FROM (SELECT n_zero_ret::DOUBLE / GREATEST(n_weeks - n_gaps, 1) AS zero_share
          FROM wkagg JOIN active USING (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)
          WHERE n_weeks >= 140)
""").df()
)
print("candidate counts above each threshold")
for thr in (0.01, 0.02, 0.03, 0.05):
    n = duckdb.sql(f"""
        SELECT count(*) FROM (SELECT n_zero_ret::DOUBLE / GREATEST(n_weeks - n_gaps, 1) AS zero_share
            FROM wkagg JOIN active USING (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)
            WHERE n_weeks >= 140) WHERE zero_share > {thr}
    """).fetchall()[0][0]
    print(f"zero_share > {thr}: {n} classes")

zero-return week share per active class with at least 140 weeks; percentiles


,p50,p90,p95,p99,p100
0,0.0,0.0064,0.0064,0.0192,0.6026


candidate counts above each threshold


zero_share > 0.01: 59 classes


zero_share > 0.02: 15 classes


zero_share > 0.03: 12 classes


zero_share > 0.05: 9 classes


- **Note:** flat weeks (zero weekly return) are rare and the distribution is tight.
  - Half of the classes never record one; p95 is 0,0064 (about one flat week in 156) and p99 is 0,019.
  - The tail is short: 59 classes above 1 %, 15 above 2 %, 12 above 3 % and 9 above 5 %.
- **Hypothesis:** a flat week comes from a padded or frozen price, the vehicle designs that `01_fondmens_fondregistro__nav_metadata_anomaly_profiling.ipynb` already classified as constant-price sentinels; a lenient cut should be enough to remove the distortion.
- **Gate:** the flat-week share is capped at 3 % for eligibility; the cap quarantines 12 classes, about six times the p95, which keeps surviving distortions scarce without deleting legitimate histories.

In [9]:
print("worst active classes by max absolute weekly return (quarantine candidates)")
display(
    duckdb.sql("""
    SELECT w.NumeroRegistro, w.NumeroCompartimento, w.NumeroClase,
           m.DenominacionClase, round(w.max_abs_ret, 3) AS max_abs_ret, w.n_weeks
    FROM (SELECT * FROM wkagg w
          JOIN active a USING (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)
          ORDER BY w.max_abs_ret DESC LIMIT 5) w
    JOIN meta m ON m.FechaDatos = '202605' AND m.Tipo = w.Tipo
        AND m.NumeroRegistro = w.NumeroRegistro
        AND m.NumeroCompartimento = w.NumeroCompartimento
        AND m.NumeroClase = w.NumeroClase
    ORDER BY max_abs_ret DESC
""").df()
)
print("extreme keys found during exploration, with active status at as_of")
display(
    duckdb.sql("""
    SELECT b.NumeroRegistro, b.NumeroCompartimento, b.NumeroClase,
           (a.NumeroRegistro IS NOT NULL) AS is_active,
           round(b.max_vl, 2) AS max_vl_seen
    FROM (SELECT Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase,
                 max(VL) AS max_vl FROM nav GROUP BY 1, 2, 3, 4) b
    LEFT JOIN active a USING (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)
    WHERE (b.Tipo, b.NumeroRegistro, b.NumeroCompartimento, b.NumeroClase)
          IN (('FI', 5138, 4, 0), ('FI', 5186, 9, 0), ('FI', 4421, 0, 0))
""").df()
)

worst active classes by max absolute weekly return (quarantine candidates)


,NumeroRegistro,NumeroCompartimento,NumeroClase,DenominacionClase,max_abs_ret,n_weeks
0,4421,0,0,CLASE 0,0.878,157
1,5316,4,1,CLASE I,0.865,65
2,5259,4,0,SERIE 0,0.280,157
3,737,8,0,SERIE 0,0.218,157
4,2855,0,0,CLASE 0,0.170,157


extreme keys found during exploration, with active status at as_of


,NumeroRegistro,NumeroCompartimento,NumeroClase,is_active,max_vl_seen
0,4421,0,0,True,9.41
1,5138,4,0,False,610.28
2,5186,9,0,False,17.15


- **Note:** with zero-VL rows excluded, only **2** active classes keep a weekly return above 50 %: `FI 4421/0/0` (0,878) and `FI 5316/4/1` (0,865; visible in the top of the ranking). The three extreme keys found during XML exploration corroborate the pattern: the first is active and lands in the quarantine pool; the other two are inactive at the as-of date and stay outside this analysis.
- **Gate:** a ceiling of 0,50 for the absolute weekly return remains the eligibility rule, and both active classes would be flagged by it.

In [10]:
print(
    "weekly vol annualized (sd × √52); SRRI-like bins over active classes with 140 or more weeks"
)
display(
    duckdb.sql("""
    SELECT srri_class, count(*) AS classes FROM (
        SELECT CASE
                   WHEN vol_annualized < 0.005 THEN 1
                   WHEN vol_annualized < 0.02  THEN 2
                   WHEN vol_annualized < 0.05  THEN 3
                   WHEN vol_annualized < 0.10  THEN 4
                   WHEN vol_annualized < 0.15  THEN 5
                   WHEN vol_annualized < 0.25  THEN 6
                   ELSE 7 END AS srri_class
        FROM (SELECT sd_w * sqrt(52) AS vol_annualized
              FROM wkagg JOIN active USING (Tipo, NumeroRegistro, NumeroCompartimento, NumeroClase)
              WHERE n_weeks >= 140))
    GROUP BY 1 ORDER BY 1
""").df()
)

weekly vol annualized (sd × √52); SRRI-like bins over active classes with 140 or more weeks


,srri_class,classes
0,1,158
1,2,285
2,3,553
3,4,472
4,5,593
5,6,206
6,7,7


- **Note:** among classes with at least 140 valid weeks, the annualized dispersion (sd × √52) bands plausibly: a small money-market box in the lowest bucket, the mass between the second and the fifth of seven, and only 7 classes in the highest.
- **Hypothesis:** this is the dispersion shape of the Spanish market, and it can be contrasted with the official quarterly volatility of [§3](#3-official-quarterly-metrics) once its unit is confirmed.
- **Gate:** the buckets are report-only; the per-vocation dispersion comparison belongs to the anomaly-detection work.

## 5. Portfolio coverage and quality
**Question:** how faithful is the reported portfolio to its compartment, and how clean are the lines? Coverage compares the value of reported positions with the compartment AUM of the same month; the band should tolerate reporting differences without demanding exactness.

In [11]:
duckdb.sql("""
    CREATE OR REPLACE TEMP VIEW cart512 AS
    SELECT Tipo, NumeroRegistro, NumeroCompartimento,
           sum(ValorMercado) AS total_value, count(*) AS n_positions
    FROM h WHERE FechaDatos = '202512'
    GROUP BY 1, 2, 3
""")
duckdb.sql("""
    CREATE OR REPLACE TEMP VIEW compnav512 AS
    SELECT Tipo, NumeroRegistro, NumeroCompartimento,
           sum(Patrimonio) AS aum, sum(Participes) AS parts
    FROM nav
    WHERE FechaDatos = '202512'
      AND value_date = (SELECT max(value_date) FROM nav WHERE FechaDatos = '202512')
    GROUP BY 1, 2, 3
""")
print(
    "coverage_ratio = reported portfolio value / compartment AUM (compartment with 20 or more holders)"
)
display(
    duckdb.sql("""
    SELECT count(*) AS compartimentos,
           round(quantile_cont(coverage_value, 0.05), 3) AS q05,
           round(quantile_cont(coverage_value, 0.50), 3) AS q50,
           round(quantile_cont(coverage_value, 0.95), 3) AS q95,
           round(quantile_cont(coverage_value, 1.0), 3) AS q100,
           sum(CASE WHEN coverage_value < 0.8 THEN 1 ELSE 0 END) AS under_08,
           sum(CASE WHEN coverage_value BETWEEN 0.8 AND 1.2 THEN 1 ELSE 0 END) AS in_band,
           sum(CASE WHEN coverage_value > 1.2 THEN 1 ELSE 0 END) AS over_12
    FROM (SELECT (c.total_value / nn.aum::DOUBLE) AS coverage_value
          FROM cart512 c JOIN compnav512 nn USING (Tipo, NumeroRegistro, NumeroCompartimento)
          WHERE nn.parts >= 20 AND nn.aum > 0)
""").df()
)

coverage_ratio = reported portfolio value / compartment AUM (compartment with 20 or more holders)


,compartimentos,q05,q50,q95,q100,under_08,in_band,over_12
0,1622,0.836,0.96,0.996,6.978,47.0,1570.0,5.0


In [12]:
print("line quality of fund_holdings across the six snapshots (one summary row)")
display(
    duckdb.sql("""
    SELECT count(*) AS positions,
           sum(CASE WHEN CodigoISIN IS NULL THEN 1 ELSE 0 END) AS isin_null,
           sum(CASE WHEN ValorMercado <= 0 THEN 1 ELSE 0 END) AS nonpositive_value,
           count(DISTINCT ClaseIF) AS n_claseif,
           count(DISTINCT Divisa) AS n_currencies,
           (SELECT count(*) FROM (
                SELECT Tipo, NumeroRegistro, NumeroCompartimento, FechaDatos,
                       CodigoISIN, ClaseIF
                FROM h GROUP BY 1, 2, 3, 4, 5, 6 HAVING count(*) > 1)
           ) AS repeated_groups,
           (SELECT count(*) FROM (
                SELECT DISTINCT Tipo, NumeroRegistro, NumeroCompartimento FROM active
                EXCEPT SELECT DISTINCT Tipo, NumeroRegistro, NumeroCompartimento
                FROM h WHERE FechaDatos = '202512')
           ) AS active_without_cart
    FROM h
""").df()
)

line quality of fund_holdings across the six snapshots (one summary row)


,positions,isin_null,nonpositive_value,n_claseif,n_currencies,repeated_groups,active_without_cart
0,572263,1296.0,38.0,3,36,2879,71


- **Note:** coverage concentrates around completeness: median 0,96 over 1.622 usable compartments, 1.570 of them (96,8 %) inside the [0,80, 1,20] band, 47 below, 5 above and a maximum of 6,98.
  - Line quality: 1.296 of 572.263 positions without ISIN (0,2 %), 38 non-positive values, 3 `ClaseIF` categories and 36 currencies quoted at position level while the market value itself is reported in euros.
  - 2.879 groups of repeated same-instrument lines exist by design and aggregate downstream.
  - 71 active compartments publish no portfolio that month, so their composition metrics are unavailable rather than wrong.
- **Hypothesis:** values above 1 report instruments outside the AUM perimeter (cash, collateral, accrued flows) and values well under 1 omit the cash leg from the snapshot; both are reporting behaviors, not parse errors.
- **Gate:** the [0,80, 1,20] band holds as the coverage rule; out-of-band compartments are annotated as `holdings_incomplete` and composition analysis keeps the declared-margin caveat.

## 6. Findings and calibration (conclusion)

Two outputs land here: the thresholds calibrated with data and the observations reported for later work. The automated gate keeps custody of structure: schema drift, duplicate and null keys, month cadence and the sign anomalies quoted in the sections above.

**Calibrated thresholds**

| Parameter | Value | Evidence |
|---|---|---|
| `max_zero_week_share` | 0,03 | p95 0,0064, p99 0,019, 12 classes quarantined ([§4](#4-weekly-series-calibration)) |
| `min_coverage` | 0,90 (confirmed) | 2.211 classes at 1,00 against the 0,98 partial-window ceiling ([§4](#4-weekly-series-calibration), [§1](#1-universe-and-publication-cadence)) |
| `extreme_return_abs` | 0,50 (confirmed) | 2 active classes beyond it ([§4](#4-weekly-series-calibration)) |
| `min_holdings_coverage` | 0,80 | p5 of the coverage distribution 0,836, 47 compartments below ([§5](#5-portfolio-coverage-and-quality)) |
| `max_holdings_coverage` | 1,20 | 5 compartments above, maximum 6,98 ([§5](#5-portfolio-coverage-and-quality)) |
| `min_peer_group_size` | 5 (confirmed) | every named vocation except `IIC de Gestión Pasiva` clears it ([§2](#2-snapshot-coverage-currencies-and-vocations)) |
| Currency field | `CodigoDivisa` at class level | the entity level carries 411 nulls and no cross-level disagreement exists ([§2](#2-snapshot-coverage-currencies-and-vocations)) |

**Reported, no calibration**
- The snapshots publish at months 06 and 12 only; the mass tables stay monthly ([§1](#1-universe-and-publication-cadence)).
- Vocation is compartment-level free text with 14 groups; guaranteed families are identified by vocation, funds-of-funds and replicas by `ClaseFondo` ([§2](#2-snapshot-coverage-currencies-and-vocations)).
- Expenses and fees arrive almost complete; the official quarterly volatility needs its unit confirmed before any cross-check ([§3](#3-official-quarterly-metrics)).
- Portfolio coverage and line quality as summarized in [§5](#5-portfolio-coverage-and-quality).

**Pre-publication checklist**

- [ ] Restart and Run All works top to bottom on a fresh kernel
- [ ] calibrated values match the executed outputs
- [ ] notebook committed with outputs (policy: exploration notebooks keep outputs so they render on GitHub)